In [0]:
from pyspark.sql.functions import *
from datetime import datetime

In [0]:
import boto3
import json
from datetime import datetime

AWS_ACCESS_KEY = ""
AWS_SECRET_KEY = ""

AWS_BUCKET_NAME = ""

s3 = boto3.client(
    "s3",
    aws_access_key_id="",
    aws_secret_access_key=""
)

response = s3.get_object(
    Bucket="",
    Key=f"raw/youtube_events_{datetime.now().date()}.json"
)

json_content = response["Body"].read().decode("utf-8")

events = json.loads(json_content)
print(events[0])

print("Downloaded JSON from S3 successfully")

{'event_time': '2026-05-28 09:20:53.245290', 'user_id': 2368, 'video_id': 'vid_3', 'video_category': 'Education', 'video_language': 'Hindi', 'country': 'India', 'device_type': 'Desktop', 'watch_time': 745, 'liked': False, 'subscribed': False}
Downloaded JSON from S3 successfully


In [0]:
raw_df = spark.createDataFrame(events)

In [0]:
raw_df.printSchema()

root
 |-- country: string (nullable = true)
 |-- device_type: string (nullable = true)
 |-- event_time: string (nullable = true)
 |-- liked: boolean (nullable = true)
 |-- subscribed: boolean (nullable = true)
 |-- user_id: long (nullable = true)
 |-- video_category: string (nullable = true)
 |-- video_id: string (nullable = true)
 |-- video_language: string (nullable = true)
 |-- watch_time: long (nullable = true)



In [0]:
raw_df.show(5, truncate=False)

+-------+-----------+--------------------------+-----+----------+-------+--------------+--------+--------------+----------+
|country|device_type|event_time                |liked|subscribed|user_id|video_category|video_id|video_language|watch_time|
+-------+-----------+--------------------------+-----+----------+-------+--------------+--------+--------------+----------+
|India  |Desktop    |2026-05-28 09:20:53.245290|false|false     |2368   |Education     |vid_3   |Hindi         |745       |
|India  |Tablet     |2026-05-28 09:20:53.245322|false|true      |7696   |Gaming        |vid_35  |Spanish       |781       |
|India  |Desktop    |2026-05-28 09:20:53.245334|true |true      |3587   |Education     |vid_45  |English       |806       |
|India  |Mobile     |2026-05-28 09:20:53.245343|true |true      |1793   |Vlogs         |vid_13  |English       |73        |
|Germany|Mobile     |2026-05-28 09:20:53.245352|false|false     |1382   |Gaming        |vid_46  |Hindi         |228       |
+-------

In [0]:
raw_df.printSchema()

root
 |-- country: string (nullable = true)
 |-- device_type: string (nullable = true)
 |-- event_time: string (nullable = true)
 |-- liked: boolean (nullable = true)
 |-- subscribed: boolean (nullable = true)
 |-- user_id: long (nullable = true)
 |-- video_category: string (nullable = true)
 |-- video_id: string (nullable = true)
 |-- video_language: string (nullable = true)
 |-- watch_time: long (nullable = true)



In [0]:
from pyspark.sql.functions import *

bronze_df = raw_df.withColumn(
    "ingestion_time",
    current_timestamp()
)

In [0]:
bronze_df.show(5, truncate=False)

+-------+-----------+--------------------------+-----+----------+-------+--------------+--------+--------------+----------+--------------------------+
|country|device_type|event_time                |liked|subscribed|user_id|video_category|video_id|video_language|watch_time|ingestion_time            |
+-------+-----------+--------------------------+-----+----------+-------+--------------+--------+--------------+----------+--------------------------+
|India  |Desktop    |2026-05-28 09:20:53.245290|false|false     |2368   |Education     |vid_3   |Hindi         |745       |2026-05-28 09:22:09.962205|
|India  |Tablet     |2026-05-28 09:20:53.245322|false|true      |7696   |Gaming        |vid_35  |Spanish       |781       |2026-05-28 09:22:09.962205|
|India  |Desktop    |2026-05-28 09:20:53.245334|true |true      |3587   |Education     |vid_45  |English       |806       |2026-05-28 09:22:09.962205|
|India  |Mobile     |2026-05-28 09:20:53.245343|true |true      |1793   |Vlogs         |vid_13

In [0]:
bronze_df.write.format("delta") \
    .option("mergeSchema", "true") \
    .mode("append") \
    .saveAsTable("bronze_youtube_events")

In [0]:
spark.sql("""
SELECT *
FROM bronze_youtube_events
LIMIT 5
""").show(truncate=False)

+-------+-----------+--------------------------+-----+----------+-------+--------------+--------+----------+--------------------------+--------------+
|country|device_type|event_time                |liked|subscribed|user_id|video_category|video_id|watch_time|ingestion_time            |video_language|
+-------+-----------+--------------------------+-----+----------+-------+--------------+--------+----------+--------------------------+--------------+
|Germany|Tablet     |2026-05-28 06:17:31.735174|true |true      |5884   |Education     |vid_46  |69        |2026-05-28 06:50:50.951196|NULL          |
|Canada |Desktop    |2026-05-28 06:17:31.735206|false|false     |4524   |AI            |vid_3   |278       |2026-05-28 06:50:50.951196|NULL          |
|Canada |Mobile     |2026-05-28 06:17:31.735218|false|true      |1090   |Vlogs         |vid_11  |96        |2026-05-28 06:50:50.951196|NULL          |
|Germany|Tablet     |2026-05-28 06:17:31.735226|true |true      |2939   |Tech          |vid_14

In [0]:
spark.sql("""
SELECT COUNT(*)
FROM bronze_youtube_events
""").show()

+--------+
|COUNT(*)|
+--------+
|    6000|
+--------+



In [0]:
bronze_export = bronze_df.toPandas()

In [0]:
bronze_export.to_csv(
    "/tmp/bronze_youtube_events.csv",
    index=False
)

In [0]:
spark.sql("""
SELECT *
FROM bronze_youtube_events
LIMIT 5
""").show(truncate=False)

+-------+-----------+--------------------------+-----+----------+-------+--------------+--------+----------+--------------------------+--------------+
|country|device_type|event_time                |liked|subscribed|user_id|video_category|video_id|watch_time|ingestion_time            |video_language|
+-------+-----------+--------------------------+-----+----------+-------+--------------+--------+----------+--------------------------+--------------+
|Germany|Tablet     |2026-05-28 06:17:31.735174|true |true      |5884   |Education     |vid_46  |69        |2026-05-28 06:50:50.951196|NULL          |
|Canada |Desktop    |2026-05-28 06:17:31.735206|false|false     |4524   |AI            |vid_3   |278       |2026-05-28 06:50:50.951196|NULL          |
|Canada |Mobile     |2026-05-28 06:17:31.735218|false|true      |1090   |Vlogs         |vid_11  |96        |2026-05-28 06:50:50.951196|NULL          |
|Germany|Tablet     |2026-05-28 06:17:31.735226|true |true      |2939   |Tech          |vid_14

In [0]:
raw_df.printSchema()

root
 |-- country: string (nullable = true)
 |-- device_type: string (nullable = true)
 |-- event_time: string (nullable = true)
 |-- liked: boolean (nullable = true)
 |-- subscribed: boolean (nullable = true)
 |-- user_id: long (nullable = true)
 |-- video_category: string (nullable = true)
 |-- video_id: string (nullable = true)
 |-- video_language: string (nullable = true)
 |-- watch_time: long (nullable = true)



In [0]:
spark.sql("""
DROP TABLE IF EXISTS bronze_youtube_events
""")

DataFrame[]

In [0]:
from pyspark.sql.functions import *

bronze_df = raw_df.withColumn(
    "ingestion_time",
    current_timestamp()
)

bronze_df.write.format("delta") \
    .option("mergeSchema", "true") \
    .mode("append") \
    .saveAsTable("bronze_youtube_events")

In [0]:
spark.sql("""
SELECT *
FROM bronze_youtube_events
LIMIT 5
""").show(truncate=False)

+-------+-----------+--------------------------+-----+----------+-------+--------------+--------+--------------+----------+--------------------------+
|country|device_type|event_time                |liked|subscribed|user_id|video_category|video_id|video_language|watch_time|ingestion_time            |
+-------+-----------+--------------------------+-----+----------+-------+--------------+--------+--------------+----------+--------------------------+
|India  |Desktop    |2026-05-28 09:20:53.245290|false|false     |2368   |Education     |vid_3   |Hindi         |745       |2026-05-28 09:24:19.259333|
|India  |Tablet     |2026-05-28 09:20:53.245322|false|true      |7696   |Gaming        |vid_35  |Spanish       |781       |2026-05-28 09:24:19.259333|
|India  |Desktop    |2026-05-28 09:20:53.245334|true |true      |3587   |Education     |vid_45  |English       |806       |2026-05-28 09:24:19.259333|
|India  |Mobile     |2026-05-28 09:20:53.245343|true |true      |1793   |Vlogs         |vid_13

In [0]:
s3.upload_file(
    "/tmp/bronze_youtube_events.csv",
    "youtube-analytics-lakehouse",
    "bronze/bronze_youtube_events.csv"
)

print("Bronze exported to S3")

Bronze exported to S3
